# AI Logistics Automation — Real Olist Data (No Kaggle API)

This Colab notebook uses the **uploaded Olist `archive.zip` directly**. No Kaggle API or Kaggle token is required.

It implements four connected decisions:

1. **Nearest eligible seller** for the customer's purchased item(s), using the official Olist geolocation table and ZIP-prefix latitude/longitude centroids.
2. **Shipping-company selection** using real Frenet quotes when `FRENET_TOKEN` is supplied; otherwise it runs in transparent DEMO mode.
3. **Product recommendations** from item-to-item co-purchase patterns learned from the customer's purchased products.
4. **Automated personalized email** containing seller, shipping company, quote/ETA, and recommended products using Resend when customer emails and `RESEND_API_KEY` are supplied.

### Real-data validation from the uploaded archive

The uploaded Olist files contain 99,441 orders, 112,650 order-item rows, 96,096 unique customers, and 1,000,163 geolocation observations. Using distinct order-level calculations, the data gives approximately **16.57% freight / product revenue, 8.11% late deliveries, and 12.56 average delivery days**.

A lightweight CatBoost validation on the full data produced about **4.04 MAE for freight prediction** and **0.78 ROC-AUC for late-delivery risk**. These are validation results from this uploaded dataset, not guaranteed production performance.


In [ ]:
!pip -q install -U catboost requests


## 1. Upload the real dataset

Upload the `archive.zip` you provided. The notebook extracts the nine Olist CSV files locally in the Colab runtime.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os, shutil, json, math, time
import numpy as np
import pandas as pd

uploaded = files.upload()

if not uploaded:
    raise RuntimeError('Upload archive.zip before continuing.')

upload_name = next(iter(uploaded.keys()))
root = Path('/content/olist_real')
if root.exists():
    shutil.rmtree(root)
root.mkdir(parents=True)

if upload_name.lower().endswith('.zip'):
    with zipfile.ZipFile(upload_name) as z:
        z.extractall(root)
else:
    raise ValueError('Please upload archive.zip containing the Olist CSV files.')

required = [
    'olist_customers_dataset.csv',
    'olist_geolocation_dataset.csv',
    'olist_order_items_dataset.csv',
    'olist_order_payments_dataset.csv',
    'olist_order_reviews_dataset.csv',
    'olist_orders_dataset.csv',
    'olist_products_dataset.csv',
    'olist_sellers_dataset.csv',
    'product_category_name_translation.csv',
]

for name in required:
    found = list(root.rglob(name))
    if not found:
        raise FileNotFoundError(f'Missing {name}')

DATA_DIR = root
print('DATA_DIR =', DATA_DIR)
for name in required:
    p = next(root.rglob(name))
    print(f'{name:45s} {p.stat().st_size/1024/1024:8.2f} MB')


## 2. Load and prepare the real Olist tables


In [ ]:
orders = pd.read_csv(DATA_DIR / 'olist_orders_dataset.csv')
items = pd.read_csv(DATA_DIR / 'olist_order_items_dataset.csv')
customers = pd.read_csv(DATA_DIR / 'olist_customers_dataset.csv')
products = pd.read_csv(DATA_DIR / 'olist_products_dataset.csv')
sellers = pd.read_csv(DATA_DIR / 'olist_sellers_dataset.csv')
geo = pd.read_csv(DATA_DIR / 'olist_geolocation_dataset.csv')
translations = pd.read_csv(DATA_DIR / 'product_category_name_translation.csv')

for c in ['order_purchase_timestamp','order_approved_at','order_delivered_customer_date','order_estimated_delivery_date']:
    orders[c] = pd.to_datetime(orders[c], errors='coerce')
items['shipping_limit_date'] = pd.to_datetime(items['shipping_limit_date'], errors='coerce')
for c in ['price','freight_value']:
    items[c] = pd.to_numeric(items[c], errors='coerce')
for c in ['product_weight_g','product_length_cm','product_height_cm','product_width_cm']:
    products[c] = pd.to_numeric(products[c], errors='coerce')

print('orders       ', orders.shape)
print('order items  ', items.shape)
print('customers    ', customers.shape)
print('products     ', products.shape)
print('sellers      ', sellers.shape)
print('geolocation  ', geo.shape)


## 3. Correct order-level business insights


In [ ]:
order_money = (
    items.groupby('order_id', as_index=False)
    .agg(
        product_revenue=('price','sum'),
        freight_cost=('freight_value','sum'),
        item_count=('order_item_id','count'),
    )
)

business = orders.merge(order_money, on='order_id', how='left')
delivered = business[
    business['order_delivered_customer_date'].notna()
    & business['order_estimated_delivery_date'].notna()
].copy()
delivered['delivery_days'] = (
    delivered['order_delivered_customer_date']
    - delivered['order_purchase_timestamp']
).dt.total_seconds()/86400
delivered['is_late'] = (
    delivered['order_delivered_customer_date']
    > delivered['order_estimated_delivery_date']
).astype(int)

revenue = float(business['product_revenue'].sum())
freight = float(business['freight_cost'].sum())
kpis = pd.DataFrame({
    'KPI': [
        'Product revenue','Freight cost','Freight / product revenue %',
        'Orders','Unique customers','Average delivery days','Late delivery %'
    ],
    'Value': [
        revenue,freight,freight/revenue*100,
        business['order_id'].nunique(),customers['customer_unique_id'].nunique(),
        delivered['delivery_days'].mean(),delivered['is_late'].mean()*100
    ]
})
display(kpis)


## 4. Build ZIP-prefix geolocation

The Olist geolocation file contains repeated latitude/longitude observations per ZIP prefix. We aggregate each prefix to a centroid, then use Haversine distance for seller/customer proximity.

For production road distance, Google Routes API can be added later; its `ComputeRouteMatrix` method is designed for many origin/destination combinations. citeturn976459search1turn976459search6


In [ ]:
geo['geolocation_zip_code_prefix'] = pd.to_numeric(geo['geolocation_zip_code_prefix'], errors='coerce')
customers['customer_zip_code_prefix'] = pd.to_numeric(customers['customer_zip_code_prefix'], errors='coerce')
sellers['seller_zip_code_prefix'] = pd.to_numeric(sellers['seller_zip_code_prefix'], errors='coerce')

centroids = (
    geo.dropna(subset=['geolocation_zip_code_prefix','geolocation_lat','geolocation_lng'])
    .groupby('geolocation_zip_code_prefix', as_index=False)
    .agg(customer_lat=('geolocation_lat','mean'), customer_lon=('geolocation_lng','mean'))
)

customers = customers.merge(centroids, left_on='customer_zip_code_prefix', right_on='geolocation_zip_code_prefix', how='left').drop(columns=['geolocation_zip_code_prefix'])
seller_centroids = centroids.rename(columns={'customer_lat':'seller_lat','customer_lon':'seller_lon'})
sellers = sellers.merge(seller_centroids, left_on='seller_zip_code_prefix', right_on='geolocation_zip_code_prefix', how='left').drop(columns=['geolocation_zip_code_prefix'])

print('Customer ZIP centroid coverage:', round(customers[['customer_lat','customer_lon']].notna().all(axis=1).mean()*100,2), '%')
print('Seller ZIP centroid coverage:  ', round(sellers[['seller_lat','seller_lon']].notna().all(axis=1).mean()*100,2), '%')


## 5. Train the AI models on the real data

The models below use CatBoost because it can handle categorical features directly without creating a huge one-hot matrix.

The late-risk model intentionally excludes final delivery date and actual delay fields from its predictive features.


In [ ]:
from catboost import CatBoostRegressor, CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, roc_auc_score, precision_recall_fscore_support

# Item-level frame for freight prediction
item_ml = (
    items.merge(products[['product_id','product_category_name','product_weight_g','product_length_cm','product_height_cm','product_width_cm']], on='product_id', how='left')
         .merge(orders[['order_id','customer_id','order_purchase_timestamp','order_approved_at','order_estimated_delivery_date']], on='order_id', how='left')
         .merge(customers[['customer_id','customer_state']], on='customer_id', how='left')
         .merge(sellers[['seller_id','seller_state']], on='seller_id', how='left')
)

item_ml['volume_cm3'] = item_ml[['product_length_cm','product_height_cm','product_width_cm']].fillna(0).prod(axis=1)
item_ml['purchase_month'] = item_ml['order_purchase_timestamp'].dt.month
item_ml['purchase_dow'] = item_ml['order_purchase_timestamp'].dt.dayofweek
item_ml['purchase_hour'] = item_ml['order_purchase_timestamp'].dt.hour
item_ml['same_state'] = (item_ml['customer_state'].fillna('UNKNOWN') == item_ml['seller_state'].fillna('UNKNOWN')).astype(int)
item_ml['days_to_shipping_limit'] = (item_ml['shipping_limit_date'] - item_ml['order_purchase_timestamp']).dt.total_seconds()/86400

FREIGHT_FEATURES = [
    'price','product_weight_g','product_length_cm','product_height_cm','product_width_cm',
    'volume_cm3','purchase_month','purchase_dow','purchase_hour','days_to_shipping_limit',
    'product_category_name','seller_state','customer_state','same_state','seller_id'
]

Xf = item_ml[FREIGHT_FEATURES].copy()
yf = item_ml['freight_value'].fillna(item_ml['freight_value'].median())
freight_cats = [c for c in FREIGHT_FEATURES if Xf[c].dtype == 'object']
for c in freight_cats:
    Xf[c] = Xf[c].fillna('UNKNOWN').astype(str)
for c in [c for c in FREIGHT_FEATURES if c not in freight_cats]:
    Xf[c] = Xf[c].fillna(Xf[c].median())

Xtr, Xte, ytr, yte = train_test_split(Xf, yf, test_size=0.20, random_state=42)
FREIGHT_MODEL = CatBoostRegressor(
    iterations=80, depth=6, learning_rate=0.10, loss_function='MAE',
    verbose=False, random_seed=42, thread_count=4
)
FREIGHT_MODEL.fit(Xtr, ytr, cat_features=freight_cats)
freight_pred = FREIGHT_MODEL.predict(Xte)
freight_mae = mean_absolute_error(yte, freight_pred)

# Order-level frame for late-delivery risk
order_agg = (
    item_ml.groupby('order_id', as_index=False)
    .agg(
        item_count=('order_item_id','count'),
        unique_products=('product_id','nunique'),
        unique_sellers=('seller_id','nunique'),
        total_price=('price','sum'),
        avg_weight_g=('product_weight_g','mean'),
        max_weight_g=('product_weight_g','max'),
        avg_volume_cm3=('volume_cm3','mean'),
        max_volume_cm3=('volume_cm3','max'),
        category_count=('product_category_name','nunique'),
        primary_category=('product_category_name', lambda s: s.dropna().mode().iloc[0] if not s.dropna().empty else 'UNKNOWN'),
        primary_seller=('seller_id', lambda s: s.dropna().mode().iloc[0] if not s.dropna().empty else 'UNKNOWN')
    )
)

order_ml = (
    orders[['order_id','customer_id','order_status','order_purchase_timestamp','order_approved_at','order_estimated_delivery_date','order_delivered_customer_date']]
    .merge(order_agg, on='order_id', how='left')
    .merge(customers[['customer_id','customer_state']], on='customer_id', how='left')
    .merge(sellers[['seller_id','seller_state']].rename(columns={'seller_id':'primary_seller','seller_state':'primary_seller_state'}), on='primary_seller', how='left')
)
order_ml = order_ml[order_ml['order_delivered_customer_date'].notna() & order_ml['order_estimated_delivery_date'].notna()].copy()
order_ml['late'] = (order_ml['order_delivered_customer_date'] > order_ml['order_estimated_delivery_date']).astype(int)
order_ml['purchase_month'] = order_ml['order_purchase_timestamp'].dt.month
order_ml['purchase_dow'] = order_ml['order_purchase_timestamp'].dt.dayofweek
order_ml['purchase_hour'] = order_ml['order_purchase_timestamp'].dt.hour
order_ml['approval_delay_hours'] = (order_ml['order_approved_at'] - order_ml['order_purchase_timestamp']).dt.total_seconds()/3600
order_ml['estimated_lead_days'] = (order_ml['order_estimated_delivery_date'] - order_ml['order_purchase_timestamp']).dt.total_seconds()/86400
order_ml['cross_state'] = (order_ml['customer_state'].fillna('UNKNOWN') != order_ml['primary_seller_state'].fillna('UNKNOWN')).astype(int)

LATE_FEATURES = [
    'item_count','unique_products','unique_sellers','total_price','avg_weight_g','max_weight_g',
    'avg_volume_cm3','max_volume_cm3','category_count','primary_category','primary_seller',
    'primary_seller_state','customer_state','purchase_month','purchase_dow','purchase_hour',
    'approval_delay_hours','estimated_lead_days','cross_state'
]
XL = order_ml[LATE_FEATURES].copy(); yl = order_ml['late']
late_cats = [c for c in LATE_FEATURES if XL[c].dtype == 'object']
for c in late_cats:
    XL[c] = XL[c].fillna('UNKNOWN').astype(str)
for c in [c for c in LATE_FEATURES if c not in late_cats]:
    XL[c] = XL[c].fillna(XL[c].median())

la, lb, ya, yb = train_test_split(XL, yl, test_size=0.20, random_state=42, stratify=yl)
LATE_MODEL = CatBoostClassifier(
    iterations=100, depth=6, learning_rate=0.10, loss_function='Logloss',
    verbose=False, random_seed=42, thread_count=4, auto_class_weights='Balanced'
)
LATE_MODEL.fit(la, ya, cat_features=late_cats)
late_prob = LATE_MODEL.predict_proba(lb)[:,1]
late_pred = (late_prob >= 0.50).astype(int)
precision, recall, f1, _ = precision_recall_fscore_support(yb, late_pred, average='binary', zero_division=0)
late_auc = roc_auc_score(yb, late_prob)

metrics = pd.DataFrame({
    'Metric':['Freight MAE','Late ROC-AUC','Late Precision','Late Recall','Late F1','Observed Late %'],
    'Value':[freight_mae,late_auc,precision,recall,f1,yl.mean()*100]
})
display(metrics)


## 6. Automated nearest-seller engine

The algorithm first looks for sellers that can fulfill the complete product basket. If none exists, it falls back to product-level assignment so one order can be split across sellers.


In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0088
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dp = np.radians(lat2-lat1); dl = np.radians(lon2-lon1)
    a = np.sin(dp/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return float(2*R*np.arctan2(np.sqrt(a), np.sqrt(1-a)))

SELLER_PRODUCTS = items.groupby('seller_id')['product_id'].apply(lambda s: set(s.astype(str))).to_dict()


def seller_distance(customer_row, seller_row):
    if pd.notna(customer_row.customer_lat) and pd.notna(customer_row.customer_lon) and pd.notna(seller_row.seller_lat) and pd.notna(seller_row.seller_lon):
        return haversine_km(customer_row.customer_lat, customer_row.customer_lon, seller_row.seller_lat, seller_row.seller_lon), 'ZIP-centroid Haversine'
    return np.nan, 'No geolocation'


def prepare_freight_features(order_row, order_items, customer_row, seller_row):
    x = order_items.copy()
    x['volume_cm3'] = x[['product_length_cm','product_height_cm','product_width_cm']].fillna(0).prod(axis=1)
    purchase_dt = order_row.order_purchase_timestamp
    x['purchase_month'] = purchase_dt.month
    x['purchase_dow'] = purchase_dt.dayofweek
    x['purchase_hour'] = purchase_dt.hour
    x['same_state'] = int(str(customer_row.customer_state) == str(seller_row.seller_state))
    x['days_to_shipping_limit'] = (x.shipping_limit_date - purchase_dt).dt.total_seconds()/86400
    x['seller_state'] = seller_row.seller_state
    x['customer_state'] = customer_row.customer_state
    x['seller_id'] = seller_row.seller_id
    X = x[FREIGHT_FEATURES].copy()
    for c in freight_cats:
        X[c] = X[c].fillna('UNKNOWN').astype(str)
    for c in [c for c in FREIGHT_FEATURES if c not in freight_cats]:
        X[c] = X[c].fillna(item_ml[c].median() if c in item_ml else 0)
    return X


def late_risk_for_seller(order_row, order_items, customer_row, seller_row, predicted_freight):
    weights = pd.to_numeric(order_items.product_weight_g, errors='coerce')
    volumes = order_items[['product_length_cm','product_height_cm','product_width_cm']].fillna(0).prod(axis=1)
    cat = order_items.product_category_name.dropna().mode().iloc[0] if not order_items.product_category_name.dropna().empty else 'UNKNOWN'
    approved = order_row.order_approved_at
    est = order_row.order_estimated_delivery_date
    purchase = order_row.order_purchase_timestamp
    row = pd.DataFrame([{
        'item_count':len(order_items), 'unique_products':order_items.product_id.nunique(), 'unique_sellers':1,
        'total_price':pd.to_numeric(order_items.price,errors='coerce').sum(), 'avg_weight_g':weights.mean(), 'max_weight_g':weights.max(),
        'avg_volume_cm3':volumes.mean(), 'max_volume_cm3':volumes.max(), 'category_count':order_items.product_category_name.nunique(),
        'primary_category':cat, 'primary_seller':seller_row.seller_id, 'primary_seller_state':seller_row.seller_state,
        'customer_state':customer_row.customer_state, 'purchase_month':purchase.month, 'purchase_dow':purchase.dayofweek, 'purchase_hour':purchase.hour,
        'approval_delay_hours':((approved-purchase).total_seconds()/3600 if pd.notna(approved) else np.nan),
        'estimated_lead_days':((est-purchase).total_seconds()/86400 if pd.notna(est) else np.nan),
        'cross_state':int(str(customer_row.customer_state)!=str(seller_row.seller_state))
    }])
    for c in late_cats: row[c]=row[c].fillna('UNKNOWN').astype(str)
    for c in [c for c in LATE_FEATURES if c not in late_cats]: row[c]=row[c].fillna(0)
    return float(LATE_MODEL.predict_proba(row[LATE_FEATURES])[:,1][0])


def rank_sellers(order_id, max_candidates=20):
    order = orders.loc[orders.order_id.eq(order_id)].iloc[0]
    customer_row = customers.loc[customers.customer_id.eq(order.customer_id)].iloc[0]
    order_items = items.loc[items.order_id.eq(order_id)].merge(
        products[['product_id','product_category_name','product_weight_g','product_length_cm','product_height_cm','product_width_cm']],
        on='product_id', how='left'
    )
    wanted = set(order_items.product_id.astype(str))
    coverage = []
    for sid, stock in SELLER_PRODUCTS.items():
        covered = len(wanted & stock)
        if covered:
            coverage.append((sid, covered, covered==len(wanted)))
    full = [x for x in coverage if x[2]]
    pool = full if full else sorted(coverage, key=lambda x:(-x[1], x[0]))[:50]
    pool_ids = [x[0] for x in pool]
    candidate_sellers = sellers[sellers.seller_id.isin(pool_ids)]
    rows=[]
    for _, s in candidate_sellers.iterrows():
        dist, method = seller_distance(customer_row, s)
        freight_features = prepare_freight_features(order, order_items, customer_row, s)
        predicted_freight = float(np.clip(FREIGHT_MODEL.predict(freight_features).sum(),0,None))
        risk = late_risk_for_seller(order, order_items, customer_row, s, predicted_freight)
        coverage_count = len(wanted & SELLER_PRODUCTS.get(s.seller_id,set()))
        rows.append({'order_id':order_id,'seller_id':s.seller_id,'seller_city':s.seller_city,'seller_state':s.seller_state,'coverage_count':coverage_count,'basket_fulfillable':coverage_count==len(wanted),'distance_km':dist,'distance_method':method,'predicted_freight':predicted_freight,'late_risk':risk})
    out=pd.DataFrame(rows)
    if out.empty:return out
    out=out.sort_values(['basket_fulfillable','distance_km','late_risk','predicted_freight'],ascending=[False,True,True,True]).reset_index(drop=True)
    return out.head(max_candidates)

# Pick a real order with several eligible sellers for the demonstration.
demo_order_id = None
for oid, g in items.groupby('order_id'):
    req=set(g.product_id.astype(str))
    elig=[sid for sid,stock in SELLER_PRODUCTS.items() if req.issubset(stock)]
    if len(req)==1 and len(elig)>3:
        demo_order_id=oid; break
if demo_order_id is None:
    demo_order_id = items.order_id.iloc[0]

seller_options = rank_sellers(demo_order_id)
display(seller_options.head(10))


## 7. Product recommendation system

The recommendation model is intentionally based on **purchased items**. It learns which products are commonly purchased in the same order and recommends products the customer has not already purchased.

This requires no external recommendation API.


In [ ]:
from itertools import combinations

INTERACTIONS = items[['order_id','product_id']].drop_duplicates().merge(orders[['order_id','customer_id']], on='order_id')
product_popularity = INTERACTIONS.groupby('product_id').size().sort_values(ascending=False)
TOP_PRODUCTS = set(product_popularity.head(1000).index)
PAIR_COUNTS = {}

for _, g in INTERACTIONS[INTERACTIONS.product_id.isin(TOP_PRODUCTS)].groupby('order_id'):
    ps = sorted(g.product_id.unique())
    for a,b in combinations(ps,2):
        PAIR_COUNTS[(a,b)] = PAIR_COUNTS.get((a,b),0) + 1

ADJ = {}
for (a,b),cnt in PAIR_COUNTS.items():
    ADJ.setdefault(a,[]).append((b,cnt))
    ADJ.setdefault(b,[]).append((a,cnt))

PRODUCT_NAMES = products[['product_id','product_category_name']].merge(
    translations, on='product_category_name', how='left'
)


def recommend_for_customer(customer_id, n=5):
    purchased=set(INTERACTIONS.loc[INTERACTIONS.customer_id.eq(customer_id),'product_id'])
    scores={}
    for pid in purchased:
        for candidate, co_count in ADJ.get(pid,[]):
            if candidate not in purchased:
                scores[candidate] = scores.get(candidate,0) + co_count
    if not scores:
        fallback=[pid for pid in product_popularity.index if pid not in purchased][:n]
        scores={pid:float(product_popularity[pid]) for pid in fallback}
    ids=[pid for pid,_ in sorted(scores.items(),key=lambda x:x[1],reverse=True)[:n]]
    out=PRODUCT_NAMES[PRODUCT_NAMES.product_id.isin(ids)].copy()
    out['recommendation_score']=out.product_id.map(scores)
    return out.sort_values('recommendation_score',ascending=False)

DEMO_CUSTOMER_ID = orders.loc[orders.order_id.eq(demo_order_id),'customer_id'].iloc[0]
recommendations = recommend_for_customer(DEMO_CUSTOMER_ID,5)
print('Demo customer:', DEMO_CUSTOMER_ID)
display(recommendations[['product_id','product_category_name_english','recommendation_score']])


## 8. Real shipping-company selection through Frenet

For Brazil, Frenet exposes a shipping quote endpoint that accepts origin CEP, destination CEP, invoice value, and `ShippingItemArray`, and returns available shipping services. citeturn582621view0turn435815search3

The notebook therefore uses the **actual customer/seller CEPs and actual product dimensions/weight from Olist**. The AI ranks returned services using price, delivery time, and predicted late risk.

Without a Frenet token, the notebook stays in DEMO mode and does not pretend the demo carrier values are real quotes.


In [ ]:
import requests
from getpass import getpass

try:
    from google.colab import userdata
    FRENET_TOKEN = userdata.get('FRENET_TOKEN') or ''
except Exception:
    FRENET_TOKEN = ''

if not FRENET_TOKEN:
    print('FRENET_TOKEN not configured -> DEMO carrier mode.')
    print('Add a Colab Secret named FRENET_TOKEN to get live carrier quotes.')
else:
    print('FRENET_TOKEN found -> LIVE Frenet quote mode.')


def make_frenet_items(order_items):
    grouped = (
        order_items.groupby([
            'product_id','product_category_name','product_weight_g',
            'product_length_cm','product_height_cm','product_width_cm'
        ], dropna=False).size().reset_index(name='Quantity')
    )
    payload=[]
    for _,r in grouped.iterrows():
        def positive(v, default=1.0):
            try: return max(float(v), default)
            except Exception: return default
        payload.append({
            'Weight': positive(r.product_weight_g,0.01)/1000.0,
            'Length': positive(r.product_length_cm,1.0),
            'Height': positive(r.product_height_cm,1.0),
            'Width': positive(r.product_width_cm,1.0),
            'Quantity': int(r.Quantity),
            'SKU': str(r.product_id),
            'Category': str(r.product_category_name if pd.notna(r.product_category_name) else 'UNKNOWN'),
            'isFragile': False,
        })
    return payload


def frenet_quote(seller_cep, recipient_cep, invoice_value, shipping_items, token):
    response = requests.post(
        'https://api.frenet.com.br/shipping/quote',
        headers={
            'accept':'application/json',
            'Content-Type':'application/json',
            'token':token,
        },
        json={
            'SellerCEP':str(seller_cep).zfill(8),
            'RecipientCEP':str(recipient_cep).zfill(8),
            'ShipmentInvoiceValue':float(invoice_value),
            'RecipientCountry':'BR',
            'ShippingServiceCode':None,
            'ShippingItemArray':shipping_items,
        },
        timeout=30,
    )
    response.raise_for_status()
    data=response.json()
    services=data.get('ShippingSevicesArray') or data.get('ShippingServicesArray') or []
    rows=[]
    for x in services:
        if x.get('Error') is True: continue
        rows.append({
            'carrier':x.get('Carrier') or x.get('ServiceDescription'),
            'carrier_code':x.get('CarrierCode'),
            'service':x.get('ServiceDescription'),
            'service_code':x.get('ServiceCode'),
            'shipping_price':float(x.get('ShippingPrice',0) or 0),
            'delivery_time_days':float(x.get('DeliveryTime',0) or 0),
            'allow_buy_label':x.get('AllowBuyLabel'),
        })
    return pd.DataFrame(rows)


def choose_carrier(quotes, late_risk):
    if quotes.empty:
        return quotes
    q=quotes.copy()
    def norm(s):
        return (s-s.min())/(s.max()-s.min()) if s.max()>s.min() else pd.Series(np.zeros(len(s)),index=s.index)
    q['cost_norm']=norm(q.shipping_price)
    q['eta_norm']=norm(q.delivery_time_days)
    q['carrier_score']=0.55*q.cost_norm + 0.20*q.eta_norm + 0.25*float(late_risk)
    return q.sort_values('carrier_score').reset_index(drop=True)


def demo_carriers(weight_kg):
    # Demo only. These are intentionally labelled synthetic until replaced by live Frenet quotes.
    demo=pd.DataFrame([
        ['DEMO_FAST','DEMO Fast Carrier',35,2],
        ['DEMO_ECO','DEMO Economy Carrier',20,6],
        ['DEMO_BAL','DEMO Balanced Carrier',26,3],
    ],columns=['carrier_id','carrier','shipping_price','delivery_time_days'])
    demo['carrier_code']='DEMO'; demo['service']=demo['carrier']; demo['service_code']='DEMO'; demo['allow_buy_label']=False
    return demo


def route_demo_order(order_id):
    order=orders.loc[orders.order_id.eq(order_id)].iloc[0]
    customer_row=customers.loc[customers.customer_id.eq(order.customer_id)].iloc[0]
    order_items=items.loc[items.order_id.eq(order_id)].merge(
        products[['product_id','product_category_name','product_weight_g','product_length_cm','product_height_cm','product_width_cm']],
        on='product_id',how='left'
    )
    seller_options=rank_sellers(order_id)
    selected_seller=seller_options.iloc[0]
    seller_row=sellers.loc[sellers.seller_id.eq(selected_seller.seller_id)].iloc[0]
    predicted_freight=float(selected_seller.predicted_freight)
    late_risk=float(selected_seller.late_risk)
    if FRENET_TOKEN:
        quotes=frenet_quote(
            seller_row.seller_zip_code_prefix,
            customer_row.customer_zip_code_prefix,
            float(pd.to_numeric(order_items.price,errors='coerce').sum()),
            make_frenet_items(order_items),
            FRENET_TOKEN
        )
        mode='LIVE FRENET'
    else:
        weight_kg=pd.to_numeric(order_items.product_weight_g,errors='coerce').fillna(0).sum()/1000
        quotes=demo_carriers(weight_kg)
        mode='DEMO'
    ranked=choose_carrier(quotes, late_risk)
    if ranked.empty:
        raise RuntimeError('No carrier/service available.')
    carrier=ranked.iloc[0]
    assignment={
        'order_id':order_id,
        'customer_id':order.customer_id,
        'customer_city':customer_row.customer_city,
        'customer_state':customer_row.customer_state,
        'seller_id':seller_row.seller_id,
        'seller_city':seller_row.seller_city,
        'seller_state':seller_row.seller_state,
        'seller_distance_km':selected_seller.distance_km,
        'predicted_freight':predicted_freight,
        'late_risk_probability':late_risk,
        'carrier':carrier.carrier,
        'service':carrier.service,
        'shipping_price':float(carrier.shipping_price),
        'delivery_time_days':float(carrier.delivery_time_days),
        'carrier_score':float(carrier.carrier_score),
        'quote_mode':mode,
    }
    return assignment, ranked

assignment, carrier_options = route_demo_order(demo_order_id)
print('Automated routing decision:')
display(pd.DataFrame([assignment]))
print('Carrier options considered:')
display(carrier_options)



## 9. Customer email automation

The Olist customer table contains **no email field**, so an actual customer email cannot be inferred from this public dataset. The notebook creates a CRM mapping template; populate it with authorized customer emails before sending.

For email delivery, Resend exposes a REST endpoint at `https://api.resend.com/emails` using a Bearer API key and `from`, `to`, `subject`, and `html` fields. citeturn252276search0turn252276search2

The notebook defaults to **DRY RUN** so it cannot accidentally email a customer list.


In [ ]:
email_template = customers[['customer_id']].drop_duplicates().copy()
email_template['email'] = ''
email_template.to_csv('/content/customer_emails_template.csv',index=False)
print('Created:', '/content/customer_emails_template.csv')
print('Fill the email column from your authorized CRM/customer system, then upload the completed CSV.')
display(email_template.head())


In [ ]:
def email_html(assignment, recommendations):
    rec_rows=[]
    for _,r in recommendations.iterrows():
        name = r.get('product_category_name_english')
        if pd.isna(name) or not name:
            name = r.get('product_category_name','recommended product')
        rec_rows.append(f'<li>{name} — product {r.product_id}</li>')
    rec_html=''.join(rec_rows)
    return f'''<html><body>
    <h2>Your order delivery plan</h2>
    <p>Order: <b>{assignment['order_id']}</b></p>
    <p>Nearest selected seller: <b>{assignment['seller_id']}</b> ({assignment['seller_city']}, {assignment['seller_state']}).</p>
    <p>Estimated seller distance: <b>{assignment['seller_distance_km']:.1f} km</b></p>
    <p>Selected shipping company/service: <b>{assignment['carrier']}</b> — {assignment['service']}.</p>
    <p>Shipping price: <b>R$ {assignment['shipping_price']:.2f}</b><br>Estimated delivery: <b>{assignment['delivery_time_days']:.0f} days</b>.</p>
    <h3>Recommended items based on your purchases</h3>
    <ul>{rec_html}</ul>
    <p><small>Recommendations come from historical purchase patterns. Carrier prices and availability can change.</small></p>
    </body></html>'''

HTML_PREVIEW = email_html(assignment, recommendations)
from IPython.display import HTML, display
print('Email preview:')
display(HTML(HTML_PREVIEW))


## 10. Send one test email or automate the customer list

To enable production email:

- Add a Colab Secret named `RESEND_API_KEY`.
- Add `EMAIL_FROM` containing a verified sender/domain.
- Upload the completed `customer_emails.csv` with `customer_id,email`.
- Keep `SEND_EMAILS=False` while testing.
- Change it to `True` only after you have verified the recipient mapping.

Resend's current API uses `POST https://api.resend.com/emails` with Bearer authentication. citeturn252276search0


In [ ]:
try:
    RESEND_API_KEY = userdata.get('RESEND_API_KEY') or ''
    EMAIL_FROM = userdata.get('EMAIL_FROM') or ''
except Exception:
    RESEND_API_KEY = ''
    EMAIL_FROM = ''

SEND_EMAILS = False
EMAIL_LIMIT = 1


def send_resend(to_email, subject, html):
    r=requests.post(
        'https://api.resend.com/emails',
        headers={
            'Authorization':f'Bearer {RESEND_API_KEY}',
            'Content-Type':'application/json',
        },
        json={
            'from':EMAIL_FROM,
            'to':[to_email],
            'subject':subject,
            'html':html,
        },
        timeout=30,
    )
    r.raise_for_status()
    return r.json()


def load_customer_emails(path):
    df=pd.read_csv(path)
    required={'customer_id','email'}
    missing=required-set(df.columns)
    if missing: raise ValueError(f'customer_emails.csv missing {sorted(missing)}')
    return df.dropna(subset=['email']).copy()

# Example production flow after uploading customer_emails.csv:
# customer_emails = load_customer_emails('/content/customer_emails.csv')
# target = customer_emails[customer_emails.customer_id.eq(assignment['customer_id'])].head(EMAIL_LIMIT)
# if SEND_EMAILS and RESEND_API_KEY and EMAIL_FROM:
#     for _,row in target.iterrows():
#         print(send_resend(row.email, f"Delivery plan for order {assignment['order_id']}", HTML_PREVIEW))
# else:
#     print('DRY RUN: no email sent.')

print('SEND_EMAILS =', SEND_EMAILS)
print('Resend configured =', bool(RESEND_API_KEY and EMAIL_FROM))


## 11. What APIs are actually needed?

| Capability | API needed? | What it does |
|---|---|---|
| Nearest seller | **No** for the Olist demo | Olist geolocation + Haversine distance are enough. |
| Road distance / traffic ETA | Optional **Google Routes API** | `ComputeRouteMatrix` can calculate route distance and duration for many origin/destination pairs. citeturn976459search1turn976459search6 |
| Real shipping-company quotes | **Frenet API** | Returns carrier/service quotes from your enabled/contracted services using origin CEP, destination CEP, invoice value, and parcel data. citeturn582621view0turn436571search1 |
| Real label / tracking workflow | **Frenet or direct carrier APIs** | Frenet documents quote, label, and tracking capabilities. citeturn436571search5 |
| Automated customer email | **Resend API** or **Gmail API** | Resend provides a simple REST email endpoint; Gmail provides `messages.send`. citeturn252276search0turn976459search0 |
| Recommendation system | **No** | It is learned locally from Olist purchase/co-purchase data. |

### Recommended architecture

`Customer order -> AI seller routing -> Frenet live quote -> AI carrier selection -> shipment -> tracking -> recommendation engine -> email service`

The one missing business input for email is a legitimate customer-email mapping because the public Olist customer table contains no email address. Do not try to infer or generate customer emails.


In [ ]:
print('Demo order:', demo_order_id)
print('\nSelected seller:', assignment['seller_id'], assignment['seller_city'], assignment['seller_state'])
print('Seller distance:', round(assignment['seller_distance_km'],1), 'km')
print('Predicted freight:', round(assignment['predicted_freight'],2))
print('Late risk:', round(assignment['late_risk_probability']*100,1), '%')
print('Carrier:', assignment['carrier'], '-', assignment['service'])
print('Quote mode:', assignment['quote_mode'])
print('\nRecommended products:')
display(recommendations[['product_id','product_category_name_english','recommendation_score']])
